# 03 Power BI Export

## 1. Project Objective

Generate analytics-ready CSV files for Power BI from confidential chatbot conversations in the loan and insurance industry. The pipeline uses deterministic cleaning, phrase matching, and regular expressions only.

## 2. Imports

In [ ]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.preprocess import load_and_preprocess
from src.keyword_search import classify_messages
from src.powerbi_export import (
    POWERBI_DIR,
    REPORT_DIR,
    RAW_PATH,
    build_campaign_summary,
    build_concern_summary,
    build_conversation_level,
    build_datetime_error_review,
    build_message_level,
    build_monthly_trend_summary,
    build_product_summary,
    export_powerbi_files,
    validate_exported_date_text,
)

## 3. Configuration

In [ ]:
raw_path = RAW_PATH
powerbi_dir = POWERBI_DIR
report_dir = REPORT_DIR

print("Raw file exists:", raw_path.exists())
print("Power BI output directory:", powerbi_dir)
print("Validation report directory:", report_dir)

## 4. Load Cleaned Data

In [ ]:
raw_df, cleaned_df = load_and_preprocess(raw_path)

print("Raw rows:", len(raw_df))
print("Cleaned rows:", len(cleaned_df))
print("Customer messages:", int(cleaned_df["is_customer_message"].sum()))

## 5. Apply Keyword Classification

In [ ]:
classified_df = classify_messages(cleaned_df)
customer_df = classified_df[classified_df["standardized_role"] == "customer"].copy()

print("Classified rows:", len(classified_df))
print("Classified customer rows:", len(customer_df))

## 6. Message-Level Results

In [ ]:
message_level = build_message_level(classified_df)

preview_columns = [
    "message_id",
    "userID",
    "time",
    "message_date",
    "month_start",
    "year_month",
    "datetime_valid",
    "industry_category",
    "product_subcategory",
    "primary_concern",
    "customer_intent",
]

message_level[preview_columns].head()

## 7. Conversation-Level Aggregation

In [ ]:
conversation_level = build_conversation_level(classified_df)

conversation_preview_columns = [
    "userID",
    "conversation_start_time",
    "conversation_end_time",
    "conversation_start_date",
    "conversation_start_datetime_valid",
    "conversation_end_datetime_valid",
    "primary_product",
    "primary_concern",
    "primary_intent",
]

conversation_level[conversation_preview_columns].head()

## 8. Expected Outcome Checks

In [ ]:
print("Main reasons customers contact the chatbot:")
display(customer_df["customer_intent"].value_counts().head(10).rename_axis("customer_intent").reset_index(name="customer_message_count"))

print("Most common loan concerns:")
display(customer_df[customer_df["industry_category"].isin(["Loan", "Loan and Insurance"])]["primary_concern"].value_counts().head(10).rename_axis("primary_concern").reset_index(name="customer_message_count"))

print("Most common insurance concerns:")
display(customer_df[customer_df["industry_category"].isin(["Insurance", "Loan and Insurance"])]["primary_concern"].value_counts().head(10).rename_axis("primary_concern").reset_index(name="customer_message_count"))

print("Specific products linked to each concern:")
display(pd.crosstab(customer_df["primary_concern"], customer_df["product_subcategory"]).head(15))

print("Campaign-related customer count:")
display(conversation_level[conversation_level["campaign_related"] == "Yes"]["userID"].nunique())

print("Campaign question types:")
display(customer_df[customer_df["is_campaign_related"] == "Yes"]["campaign_question_type"].value_counts().rename_axis("campaign_question_type").reset_index(name="customer_message_count"))

print("Customers showing intention to join a campaign:")
display(conversation_level[conversation_level["campaign_joining_intent"] == "Yes"]["userID"].nunique())

print("Campaigns related to loans and insurance:")
display(customer_df[customer_df["is_campaign_related"] == "Yes"].groupby(["campaign_name", "campaign_related_product"]).size().reset_index(name="customer_message_count").head(20))

print("Main product-condition questions:")
display(customer_df[customer_df["primary_concern"].isin(["Product Conditions", "Eligibility", "Required Documents"])]["primary_concern"].value_counts().rename_axis("primary_concern").reset_index(name="customer_message_count"))

print("Payment concerns by product:")
payment_concerns = ["Monthly Installment", "Payment Method", "Late Payment", "Outstanding Balance", "Early Repayment", "Debt or Collection", "Insurance Premium"]
display(customer_df[customer_df["primary_concern"].isin(payment_concerns)].groupby(["primary_concern", "product_subcategory"]).size().reset_index(name="customer_message_count").head(20))

print("Main system or chatbot problems:")
display(customer_df[customer_df["primary_concern"] == "Account or System Issue"]["customer_intent"].value_counts().rename_axis("customer_intent").reset_index(name="customer_message_count"))

## 9. Summary Tables

In [ ]:
concern_summary = build_concern_summary(classified_df)
product_summary = build_product_summary(classified_df)
campaign_summary = build_campaign_summary(classified_df)
monthly_trend_summary = build_monthly_trend_summary(classified_df)

display(concern_summary.head(10))
display(product_summary.head(10))
display(campaign_summary.head(10))
display(monthly_trend_summary.head(10))

## 10. Data Quality Validation

In [ ]:
unknown_concern_pct = round((customer_df["primary_concern"] == "Unknown").mean() * 100, 2)
unknown_product_pct = round((customer_df["product_subcategory"] == "Unknown").mean() * 100, 2)
campaign_related_pct = round((customer_df["is_campaign_related"] == "Yes").mean() * 100, 2)

validation_checks = pd.DataFrame({
    "check": [
        "duplicate_message_ids",
        "invalid_dates",
        "unknown_concern_percentage",
        "no_product_match_percentage",
        "campaign_related_percentage"
    ],
    "value": [
        int(classified_df["message_id"].duplicated().sum()),
        int(classified_df["time"].isna().sum()),
        unknown_concern_pct,
        unknown_product_pct,
        campaign_related_pct
    ]
})

validation_checks

## Datetime Parsing Check

Review parsed date coverage before export. Invalid message times are retained and flagged instead of silently dropping rows.

In [ ]:
parsed_time = pd.to_datetime(classified_df["time"], errors="coerce")

datetime_parsing_check = pd.DataFrame({
    "metric": [
        "total_message_rows",
        "valid_time_rows",
        "invalid_or_missing_time_rows",
        "minimum_parsed_time",
        "maximum_parsed_time",
    ],
    "value": [
        len(classified_df),
        int(parsed_time.notna().sum()),
        int(parsed_time.isna().sum()),
        parsed_time.min().strftime("%Y-%m-%d %H:%M:%S") if parsed_time.notna().any() else "",
        parsed_time.max().strftime("%Y-%m-%d %H:%M:%S") if parsed_time.notna().any() else "",
    ]
})

datetime_parsing_check

## Invalid Datetime Review

This review table is exported for local quality control. It may contain confidential customer text.

In [ ]:
datetime_error_review = build_datetime_error_review(classified_df, conversation_level)

review_preview_columns = [
    "source_table",
    "message_id",
    "userID",
    "original_time_value",
    "datetime_field",
    "parse_status",
]

datetime_error_review[review_preview_columns].head(20)

## Export Corrected CSV Files

Write UTF-8 with BOM CSV files using empty fields for missing datetime values.

In [ ]:
output_paths = export_powerbi_files(raw_path)

for name, export_path in output_paths.items():
    print(f"{name}: {export_path}")

## Power BI Date Format Validation

Reopen the exported CSV files as text and verify nonblank date fields use ISO-style formats.

In [ ]:
date_text_checks = validate_exported_date_text(output_paths)

pd.DataFrame(
    [
        {"check": key, "value": value}
        for key, value in date_text_checks.items()
        if key != "invalid_date_examples"
    ]
)

In [ ]:
invalid_date_examples = date_text_checks.get("invalid_date_examples", [])

if invalid_date_examples:
    pd.DataFrame({"invalid_date_example": invalid_date_examples})
else:
    pd.DataFrame({"invalid_date_example": ["No nonblank exported date-format violations found."]})

## 12. Recommended Power BI Visualizations

### Page 1: Executive Overview
- Total customer messages
- Unique customers
- Loan-related customers
- Insurance-related customers
- Campaign-related customers
- Top five customer concerns
- Top five customer intents

### Page 2: Product Demand
- Customer count by industry
- Customer count by product category
- Product and concern matrix
- Top product-condition questions
- Monthly product-demand trend

### Page 3: Campaign Analysis
- Customers by campaign
- Campaign question type
- Joining intent
- Campaign-related product
- Campaign and concern matrix

### Page 4: Customer Concerns
- Concern frequency
- Concern by product
- Payment concern by product
- Application concern by product
- Insurance coverage and claim concerns

### Page 5: Conversation Review
- Searchable detailed message table
- Filters for product, concern, campaign, intent, date, and confidence